In [ ]:
# from torch_geometric.loader import DataLoader
from torch_geometric.data import Batch
from torch.utils.data import DataLoader
import torch
import sys
import os

sys.path.append('..')

from src.sparse_dataloader import TrafficDataset
import h5py
import numpy as np
import pandas as pd

In [ ]:
# def stack_collate(batch):
#     # batch: list[torch_geometric.data.Data] with identical shapes per field
#     x_static  = torch.stack([b.x_static  for b in batch], dim=0)  # (B, N, F_static)
#     x_dynamic = torch.stack([b.x_dynamic for b in batch], dim=0)  # (B, N, Tin, F_dyn)
#     y         = torch.stack([b.y         for b in batch], dim=0)  # (B, N, Tout)

#     # edge_index is identical for all items → repeat to get (B, 2, E)
#     edge_index = batch[0].edge_index.unsqueeze(0).repeat(len(batch), 1, 1)

#     return {
#         "edge_index": edge_index,       # (B, 2, E)
#         "x_static": x_static,           # (B, N, F_static)
#         "x_dynamic": x_dynamic,         # (B, N, Tin, F_dyn)
#         "y": y,                         # (B, N, Tout)
#     }

In [ ]:
def pad_collate(batch):
    # each item is a torch_geometric.data.Data from your dataset (subgraph of nodes)
    B = len(batch)
    # Node/edge sizes per item
    Ns = [d.x_static.size(0)   for d in batch]
    Es = [d.edge_index.size(1) for d in batch]

    Nmax, Emax = max(Ns), max(Es)

    # Infer feature dims (all items share dims)
    Fstat = batch[0].x_static.size(1)
    Tin   = batch[0].x_dynamic.size(1)
    Fdyn  = batch[0].x_dynamic.size(2)
    Tout  = batch[0].y.size(1)

    # Allocate padded tensors
    x_static  = torch.zeros(B, Nmax, Fstat,  dtype=batch[0].x_static.dtype)
    x_dynamic = torch.zeros(B, Nmax, Tin, Fdyn, dtype=batch[0].x_dynamic.dtype)
    y         = torch.zeros(B, Nmax, Tout, dtype=batch[0].y.dtype)
    node_mask = torch.zeros(B, Nmax, dtype=torch.bool)

    edge_index = torch.zeros(B, 2, Emax, dtype=batch[0].edge_index.dtype)
    edge_mask  = torch.zeros(B, Emax, dtype=torch.bool)

    # Copy each item into the padded tensors
    for i, d in enumerate(batch):
        n, e = d.x_static.size(0), d.edge_index.size(1)
        x_static[i, :n]      = d.x_static
        x_dynamic[i, :n]     = d.x_dynamic
        y[i, :n]             = d.y
        node_mask[i, :n]     = True

        edge_index[i, :, :e] = d.edge_index
        edge_mask[i, :e]     = True

    return {
        "edge_index": edge_index,  # (B, 2, Emax)
        "edge_mask":  edge_mask,   # (B, Emax) -> which edges are real
        "x_static":   x_static,    # (B, Nmax, F_static)
        "x_dynamic":  x_dynamic,   # (B, Nmax, Tin, F_dyn)
        "y":          y,           # (B, Nmax, Tout)
        "node_mask":  node_mask,   # (B, Nmax)  -> which nodes are real
        "Nmax":       Nmax,
        "Emax":       Emax,
    }

In [ ]:
processed_data_dir = "../data/datos_procesados"
sparse_data_dir = f"{processed_data_dir}/sparse_datasets"
graph_filename = f"{processed_data_dir}/conectivity_graph/road_graph.pickle"
static_data_filename = f"{sparse_data_dir}/static_dataset.h5"
dynamic_data_filenames = [
    f"{sparse_data_dir}/coi_points_dataset.h5",
    f"{sparse_data_dir}/coi_tracts_dataset.h5",
    f"{sparse_data_dir}/irregularities_dataset.h5",
    f"{sparse_data_dir}/events_dataset.h5",
    f"{sparse_data_dir}/incidents_dataset.h5",
]
target_data_filename = f"{sparse_data_dir}/jams_dataset.h5"
in_timesteps = 6
out_timesteps = 3

traffic_dataset = TrafficDataset(
    graph_path=graph_filename,
    static_data_path=static_data_filename,
    dynamic_data_paths=dynamic_data_filenames,
    target_data_path=target_data_filename,
    in_timesteps=in_timesteps,
    out_timesteps=out_timesteps,
    node_batch_size=1024,
)


In [ ]:
batch_size = 2

# traffic_dataloader = DataLoader(
#     dataset=traffic_dataset,
#     batch_size=batch_size, # This now batches the Data objects
#     shuffle=True,
#     num_workers=0
# )

# traffic_dataloader = DataLoader(
#     traffic_dataset,
#     batch_size=8,
#     shuffle=True,
#     collate_fn=stack_collate,
# )

# traffic_dataloader = DataLoader(
#     traffic_dataset,            # node_batch_size=None in the dataset
#     batch_size=batch_size,
#     shuffle=True,
#     num_workers=0,
#     collate_fn=stack_collate,   # ← key part
# )

traffic_dataloader = DataLoader(
    traffic_dataset,           # created with node_batch_size=... (your code supports this) :contentReference[oaicite:3]{index=3}
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    collate_fn=pad_collate,
)

In [ ]:
batch = next(iter(traffic_dataloader))
print("Edge index:", batch["edge_index"].shape)     # (B, 2, E)
print("Static:",     batch["x_static"].shape)       # (B, N, F_static)
print("Dynamic:",    batch["x_dynamic"].shape)      # (B, N, Tin, F_dyn)
print("Target:",     batch["y"].shape)

In [ ]:
print(f"\nDataset size: {len(traffic_dataset)} samples")
print(f"Number of batches: {len(traffic_dataloader)}")
print("-" * 40)
print("-" * 40)
# Get a single batch
sample_batch = next(iter(traffic_dataloader))

# Inspect the batch
print(f"Edge index shape (batch_size, 2, num_edges):")
print(f"{sample_batch['edge_index'].shape}")
print()
print(f"Static features shape (batch_size, num_nodes, num_static_features):")
print(f"{sample_batch['x_static'].shape}")
print()
print(f"Dynamic features shape (batch_size, num_nodes, in_timesteps, num_dynamic_features):")
print(f"{sample_batch['x_dynamic'].shape}")
print()
print(f"Target shape (batch_size, num_nodes, out_timesteps):")
print(f"{sample_batch['y'].shape}")

In [ ]:
print(f"Edge index shape (batch_size, 2, num_edges):")
print(f"Static features shape (batch_size, num_nodes, num_static_features):")
print(f"Dynamic features shape (batch_size, num_nodes, in_timesteps, num_dynamic_features):")
print(f"{sample_batch['x_dynamic'].shape}")
print(f"Target shape (batch_size, num_nodes, out_timesteps):")

In [ ]:
sample_batch['x_static'].shape

In [ ]:
file_path = traffic_dataset.dynamic_data_paths[2]
idx = 100


file_indices = np.zeros((traffic_dataset.total_timesteps_in_data - 1, 2), dtype=int)
with h5py.File(file_path, "r") as f:
    # Reading the entire time_idx column can be slow
    time_column = f["data"][:, 1]
    file_indices[:, 0] = np.searchsorted(time_column, np.arange(traffic_dataset.total_timesteps_in_data - 1), side='left')
    file_indices[:, 1] = np.searchsorted(time_column, np.arange(traffic_dataset.total_timesteps_in_data - 1), side='right')


In [ ]:
def demo_sorted(idx):
    input_start_time = idx
    input_end_time = input_start_time + 50
    with h5py.File(file_path, "r") as f:
        start_index = file_indices[input_start_time, 0]
        end_index = file_indices[input_end_time, 1]
        relevant_data = f["data"][start_index:end_index]
    return relevant_data

In [ ]:
relevant_data_sorted = demo_sorted(idx)

In [ ]:
rows, cols = 100, 50
data = np.zeros((rows, cols), dtype=int)

# Randomly assign some nonzero values
np.random.seed(42)
num_nonzero = 50
row_idx = np.random.randint(0, rows, num_nonzero)
col_idx = np.random.randint(0, cols, num_nonzero)
values = np.random.randint(1, 100, num_nonzero)
data[row_idx, col_idx] = values

In [ ]:
df = pd.DataFrame(data, columns=range(1, cols+1))

In [ ]:
df.head(14).style.map(lambda x: 'color: red' if x != 0 else '', subset=df.columns)

In [ ]:
batch_size = 2

# IMPORTANT: With this lazy-loading approach, using multiple workers (num_workers > 0)
# can be tricky due to how file handles are passed to subprocesses
# If multiprocessing is needed, we may need to handle file opening within each worker process
traffic_dataloader = DataLoader(
    traffic_dataset, batch_size=batch_size, shuffle=True, num_workers=0
)


In [ ]:
print(f"\nDataset size: {len(traffic_dataset)} samples")
print(f"Number of batches: {len(traffic_dataloader)}")
print("-" * 40)
print("-" * 40)
# Get a single batch
sample_batch = next(iter(traffic_dataloader))

In [ ]:
print(f"Edge index shape (batch_size, 2, num_edges):")
print(f"{sample_batch['edge_index'].shape}")
print()
print(f"Static features shape (batch_size, num_nodes, num_static_features):")
print(f"{sample_batch['x_static'].shape}")
print()
print(
    f"Dynamic features shape (batch_size, num_nodes, in_timesteps, num_dynamic_features):"
)
print(f"{sample_batch['x_dynamic'].shape}")
print()
print(f"Target shape (batch_size, num_nodes, out_timesteps):")
print(f"{sample_batch['y'].shape}")